# User Type Prediction - Real Estate Ads Analysis

### Section Overview:
**This module is a core component** of a larger real estate analytics platform, specifically designed to automatically classify users into **Real Estate Agents** or **Personal Users** based on their advertisement text patterns.

### Module Objectives:
- **Text Analysis**: Process Persian real estate advertisement text
- **Feature Extraction**: Convert text to numerical representations using FastText
- **Classification**: Predict user type using multiple machine learning models
- **Performance Evaluation**: Compare model accuracy and efficiency

### Technical Implementation:

#### Data Processing Flow:
Raw Text → Persian NLP Processing → FastText Embeddings → Document Vectors → Classification Models

#### Key Components:
1. **Text Preprocessing**
   - Persian character normalization
   - Tokenization and lemmatization using Hazm
   - Stopword removal and text cleaning

2. **Feature Engineering**
   - FastText word embeddings (200 dimensions)
   - Document vectorization via average word vectors
   - Vocabulary size: 12,641 Persian words

3. **Classification Models**
   - Logistic Regression
   - Random Forest
   - Linear SVM
   - SGD Classifier

### Module Performance:
- **Best Model**: Linear SVM
- **Accuracy**: 85.60%
- **Training Data**: 60,000 balanced samples
- **Feature Dimension**: 200

### Business Value:
- Enables user type-based personalization
- Supports targeted marketing campaigns
- Provides insights for platform optimization
- Enhances user experience through automated profiling

---
**Note**: This is one specialized component within a comprehensive real estate analytics platform, focusing specifically on user type classification through text analysis.

## Section 1: Environment Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
from gensim.models import FastText
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from collections import Counter
import fasttext
import os
import string
import time
warnings.filterwarnings("ignore")

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


Explanation:

Import essential libraries for data manipulation, machine learning and NLP

pandas for data handling, numpy for numerical operations

gensim for FastText word embeddings

sklearn for machine learning models and evaluation metrics

warnings to suppress unnecessary warnings

## Section 2: Persian NLP Setup

In [ ]:
!pip install git+https://github.com/sobhe/hazm.git
from hazm import *
normalizer = Normalizer()
print(normalizer.normalize("سلاممم  دنیااا"))

Explanation:

-Install and import Hazm, the primary Persian NLP library

-Normalizer() standardizes Persian text (removes extra characters, fixes spacing)

-Test the normalizer with a sample Persian text

In [ ]:
!pip show hazm

## Section 3: Data Loading and Exploration

In [ ]:
df = pd.read_csv(DATA_RAW / 'real_estate_ads.csv')

In [ ]:
df.shape

In [ ]:
df_txt = df[['description', 'title', 'cat3_slug', 'user_type']].copy()
print(df_txt["cat3_slug"].value_counts())
print(df["user_type"].value_counts())

In [ ]:
print(df_txt.columns)
df_txt.info()

Explanation:

Load the real estate ads dataset from CSV file

Check dataset dimensions (1M rows, 60 columns)

Select relevant text columns for analysis

Display category distribution and user type counts

Key Insight: Severe class imbalance (256K agents vs 33K personal users)

## Section 4: Data Cleaning and Preparation

In [ ]:
df_txt.drop_duplicates(inplace=True)
df_txt.dropna(subset=['description', 'title'], inplace=True)
print(df_txt.shape)

In [ ]:
df_txt['clean_text'] = df_txt['description'] + ' ' + df_txt['title']

Explanation:

Remove duplicate records to avoid bias

Drop rows with missing text data

Combine description and title into a single text field for processing

Final dataset: 996,828 clean records

## Section 5: Advanced Text Preprocessing

In [ ]:
word_counts = Counter(" ".join(df_txt['clean_text']).split())
[word for word, c in word_counts.most_common(200)]

### 5.1 Character Normalization Function

In [ ]:
def preprocess_text(text):
    if text is None:
        return ""
    text = str(text)
    # Persian-Arabic character standardization
    replacements = {
        'ك': 'ک', 'دِ': 'د', 'بِ': 'ب', 'زِ': 'ز', 'ذِ': 'ذ', 'شِ': 'ش', 'سِ': 'س', 'ى': 'ی',
        'ي': 'ی', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9', '٠': '0',
        '۱': '1', '۲': '2', '۳': '3', '۴': '4', '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9', '۰': '0'
    }

    # Apply character replacements
    for k, v in replacements.items():
        text = text.replace(k, v)

    # Remove zero-width and control characters
    zero_width_chars = [
        "\u200c", "\u200b", "\u200d", "\uFEFF", "\u2060",
        "\u202F", "\u200f", "\u202a", "\u202e"
    ]
    for zw in zero_width_chars:
        text = text.replace(zw, " ")

    # Clean social media symbols and special characters
    text = re.sub(r"[#@]", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\u0600-\u06FF\s]", " ", text)

    # Reduce letter repetitions
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # Normalize units
    text = re.sub(r"(\d+)\s+(متر|اتاق|سال|طبقه|خواب)", r"\1\2", text)

    # Clean extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# --- Function test ---
sample = "فروش فوری #ویلا ی 120   متر در خیابان  آزادی!!! @admin تلفن: ۰۹۱۲۳۴۵۶۷۸۹ خیلیییی تمیز"
print(preprocess_text(sample))

Explanation:

Character Standardization: Convert Arabic characters to Persian equivalents

Number Normalization: Standardize different numeral systems

Noise Removal: Eliminate zero-width characters and control codes

Pattern Cleaning: Handle repeated letters and unit formatting

Output: Clean, standardized Persian text ready for NLP

### 5.2 Complete Text Cleaning Pipeline

In [ ]:
# Load Persian stopwords
stopwords = pd.read_csv(DATA_RAW / 'Stopwords.csv', header=0)
stops = set(stopwords['word'])
stops.update([
    'تماس', 'شما', 'باسلام', 'سلام', 'درود', 'هماهنگی', 'مشاهده', 'آگهی',
    'کلیک', 'اطلاعات', 'لطفا', 'جهت', 'بازدید', 'مورد', 'بابت'
])
# normalizer = Normalizer()
tokenizer = WordTokenizer()
lemmatizer = Lemmatizer()


def clean_text(raw_text):
    text = preprocess_text(raw_text)
    words = tokenizer.tokenize(text)

    meaningful_words = []
    for w in words:
        lemma = lemmatizer.lemmatize(w)

        # Handle Hazm lemmatization artifacts
        if '#' in lemma:
            lemma = lemma.replace('#', ' ')

        # Filter stopwords
        if (lemma not in stops) and (w not in stops):
            meaningful_words.append(lemma)
    return " ".join(meaningful_words)


Explanation:

Stopword Management: Load base stopwords and add domain-specific terms

Tokenization: Split text into individual words using Hazm tokenizer

Lemmatization: Reduce words to their root forms

Artifact Handling: Fix Hazm's lemmatization output format

Stopword Filtering: Remove common and domain-specific stopwords

In [ ]:
df_txt['clean_text'] = df_txt['clean_text'].apply(clean_text)


In [ ]:
# Finding Stop Words
word_counts = Counter(" ".join(df_txt['clean_text']).split())
[word for word, c in word_counts.most_common(200)]

## Section 6: Data Balancing and FastText Preparation

In [ ]:
# Remove records with missing user_type
df_txt_cleaned = df_txt.dropna(subset=['user_type']).copy()

# Balanced sampling - 30K from each class
N = 30000

df_small = (
    df_txt_cleaned
    .groupby("user_type")
    .apply(lambda x: x.sample(min(len(x), N), random_state=42))
    .reset_index(drop=True)
)

print(df_small["user_type"].value_counts())
print("تعداد کل نمونه‌ها:", len(df_small))

# Prepare sentences for FastText training
sentences = [text.split() for text in df_small['clean_text']]
print(f"تعداد کل نمونه‌ها: {len(sentences)}")
print(f"تعداد کلمات در اولین سند: {len(sentences[0])}")

# Format data for supervised FastText
# df_txt_cleaned["ft_format"] = '__label__' + df_txt_cleaned["cat3_slug"].astype(str) + ' ' + df_txt_cleaned["clean_text"].astype(str)
df_small["ft_format"] = '__label__' + df_small["user_type"].astype(str) + ' ' + df_small["clean_text"].astype(str)

# Train-test split with stratification
# train_df, test_df = train_test_split(df_txt_cleaned, test_size=0.2, random_state=42, stratify=df_txt_cleaned["cat3_slug"])
train_df, test_df = train_test_split(df_small, test_size=0.2, random_state=42, stratify=df_small["user_type"])

train_file = 'fasttext_train.txt'
test_file = 'fasttext_test.txt'

train_df['ft_format'].to_csv(train_file, index=False, header=False, encoding='utf-8')
test_df['ft_format'].to_csv(test_file, index=False, header=False, encoding='utf-8')

print(f"فایل آموزش در مسیر {train_file} ذخیره شد. تعداد نمونه‌ها: {len(train_df)}")


Explanation:

Class Imbalance Handling: Sample 30K records from each user type

Stratified Sampling: Maintain class distribution in train/test splits

FastText Format: Convert to FastText supervised learning format (__label__value text)

Data Split: 80% training (48K samples), 20% testing (12K samples)

## Section 7: Word Embedding Training

In [ ]:
# import multiprocessing
# WORKERS = multiprocessing.cpu_count()
VECTOR_SIZE = 200
MIN_COUNT = 5
WORKERS = 8
EPOCHS = 10

print(f"\nدر حال آموزش مدل FastText با ابعاد {VECTOR_SIZE}...")
start_time = time.time()

fasttext_model = FastText(
    sentences,
    vector_size=VECTOR_SIZE,
    window=5,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=1,
    epochs=EPOCHS 
)

training_time = time.time() - start_time
print(f"آموزش FastText در زمان {training_time:.2f} ثانیه به پایان رسید.")
print(f"اندازه واژگان مدل: {len(fasttext_model.wv)}")

Explanation:

Embedding Dimensions: 200-dimensional word vectors

Training Parameters:

window=5: Context window size

min_count=5: Ignore rare words

sg=1: Use Skip-gram algorithm (better for semantic relationships)

workers=8: Parallel processing

Output: Trained FastText model with 12,641 Persian words in vocabulary

## Section 8: Document Vectorization

In [ ]:
def document_vector(model, doc_words):
    # Filter words present in vocabulary
    words = [word for word in doc_words if word in model.wv]
    
    if len(words) >= 1:
        # Average of word vectors
        return np.mean(model.wv[words], axis=0)
    else:
        # Zero vector for empty documents
        return np.zeros(model.vector_size)
print("تبدیل اسناد به بردار میانگین...")
X = np.array([document_vector(fasttext_model, doc) for doc in sentences])
# y = df_txt_cleaned["cat3_slug"]
y = df_small["user_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"ابعاد داده‌های آموزشی: {X_train.shape}")
print(f"ابعاد داده‌های تست: {X_test.shape}")

Explanation:

Document Representation: Convert each document to average of its word vectors

Vocabulary Filtering: Only use words present in FastText vocabulary

Zero Vector Handling: Return zero vector if no valid words found

Final Dataset: 48K training vectors, 12K test vectors (200 dimensions each)

## Section 9: Model Training and Evaluation

In [ ]:
# Encode string labels to numerical values
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Define multiple classification models

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        class_weight='balanced',
        max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=WORKERS
    ),
    'Linear SVM': SVC(
        kernel='linear',
        random_state=42,
        class_weight='balanced',
    ),
    'SGD Classifier (Optimized)': SGDClassifier(
    loss='hinge',
    alpha=0.0001,
    max_iter=1000, 
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)
}

# Train and evaluate all models

results = {}

for name, model in models.items():
    print(f"\n--- آموزش مدل {name} ---")
    start_time_model = time.time()
    # Model training
    model.fit(X_train, y_train_enc)
    
    # Predicting
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test_enc, y_pred)
    report = classification_report(y_test_enc, y_pred)
    cm = confusion_matrix(y_test_enc, y_pred)
    
    results[name] = {
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': cm
    }
    
    print(f"دقت مدل {name}: {accuracy:.4f}")
    print(f"زمان اجرا: {time.time() - start_time_model:.2f} ثانیه")
    print(report)

Explanation:

Label Encoding: Convert "مشاور املاک"/"شخصی" to 0/1

Model Variety: Test different algorithm types (linear, tree-based, optimized)

Class Balancing: All models use class_weight='balanced' for imbalance handling

Comprehensive Evaluation: Accuracy, classification report, confusion matrix

Performance Tracking: Measure both accuracy and training time

## Section 10: Results Analysis

In [ ]:
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
print(f"بهترین مدل: {best_model_name} با دقت {results[best_model_name]['accuracy']:.4f}")

Explanation:

Model Comparison: Find model with highest accuracy

Best Performer: Linear SVM with 85.60% accuracy

Key Insight: Linear models outperformed tree-based methods for this text classification task
